# ПРОЕКТ: Нейросеть для автопродления текстов
## Загрузка и первичный просмотр данных

In [ ]:
# =========================
# Standard Library
# =========================

import random

# =========================
# Third-Party Imports
# =========================

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split

from datasets import Dataset

import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

from transformers import (AutoTokenizer,
                          PreTrainedTokenizerBase,
                          AutoModelForCausalLM,
                          logging)

# =========================
# Local Imports
# =========================

# классы датасетов
from src.train_eval_datasets import TrainDataset, EvalDataset

# модель LSTM
from src.lstm_model import SequencePredictionLSTM

# функция обучения
from src.lstm_train import train

# вспомогательные функции
from src.utils import (get_device, # выбор 'cpu', 'mps' или 'cuda'
                       clean_string_lstm, # чистка для LSTM
                       clean_string_transformer, # чистка для transformer (менее агресивная)
                       save_model_dict, # сохранение весов для LSTM модели
                       load_model_params_from_dict, # загрузка весов для LSTM модели
                       drop_short_texts, # удаление коротких текстов менее min_len
                       drop_long_texts, # удаление длинных текстов более max_len
                       split_text) # разделение текста на 3/4 (input) и 1/4 (target)

# загрузка данных
from src.data_download import load_sentiment140_csv

# функции для оценки качества моделей
from src.evaluation import (evaluate_rouge_transformer,
                            lstm_generate_text_examples,
                            transformer_generate_text_examples)

# =========================
# Constants / Config
# =========================

# отношения размерностей выборок
TEST_VAL_SIZE = 0.2
TEST_SIZE = 0.5

# параметры обучения
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 10

# параметры моделей
MODEL_NAME = 'distilgpt2'
LSTM_HIDDEN_STATE = 128
PAD_TOKEN_ID = 0

# фиксация генераторов случайных чисел для воспроизводимости
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# опции вывода таблиц pandas
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

# отключение лишнего логгирования
logging.set_verbosity_error()

In [ ]:
# загрузка/сохранение датасета
data = load_sentiment140_csv()

In [ ]:
# первичный просмотр
data.head()
data.info()

1600000 текстов. Нас интересует только колонка `text` (сами тексты).

In [ ]:
# выбираем тексты
data = data[['text']]

## Анализ количества слов в текстах

In [ ]:
# статистика по количеству слов
word_counts = [len(t.split()) for t in data['text'].values]
print("\nСтатистика по количеству слов в тексте:")
print("Среднее: ", np.mean(word_counts))# посчитайте среднее кол-во слов в текстах
print("Медиана: ", np.median(word_counts))# посчитайте медианное кол-во слов в текстах
print("5-й перцентиль: ", np.percentile(word_counts, 5))# посчитайте 5й перцентиль кол-ва слов в текстах
print("95-й перцентиль: ", np.percentile(word_counts, 95))# посчитайте 95й перцентиль кол-ва слов в текстах

In [ ]:
# гистограмма распределения длины
plt.hist(word_counts, bins=40, edgecolor='black')
plt.title("Распределение количества слов в текстах")
plt.xlabel("Количество слов")
plt.ylabel("Частота")
plt.grid(True)
plt.show()

Имеет смысл ограничиться длинами от 3 до 26 слов.

In [ ]:
min_length = 3
max_length = 26
max_length_with_eos = max_length + 1 # длина текста с учетом токена окончания последовательности

## Деление выборки

In [ ]:
train_texts, val_test_texts = train_test_split(
    data.iloc[0:10000], test_size=TEST_VAL_SIZE, random_state=RANDOM_STATE
)
val_texts, test_texts = train_test_split(
    val_test_texts, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

In [ ]:
print('Train size:', len(train_texts))
print('Validation size:', len(val_texts))
print('Test size:', len(test_texts))

## Модель на основе LSTM

In [ ]:
# чистка текстов (для LSTM)
train_texts_cleaned = train_texts['text'].apply(clean_string_lstm)
val_texts_cleaned = val_texts['text'].apply(clean_string_lstm)

In [ ]:
# отсекаем тексты длиной менее 3 слов (важно это делать после чистки)
train_texts_cleaned = drop_short_texts(train_texts_cleaned, min_length)
val_texts_cleaned = drop_short_texts(val_texts_cleaned, min_length)

In [ ]:
# предобученный токенайзер
tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# создаем кастомные датасеты с встроенной токенизацией
train_dataset = TrainDataset(
    texts=train_texts_cleaned.tolist(),
    tokenizer=tokenizer,
    max_length=max_length
)

val_dataset = EvalDataset(
    texts=val_texts_cleaned.tolist(),
    tokenizer=tokenizer,
    max_length=max_length
)

In [ ]:
# функция для обработки токенизированного батча в даталоадере
def collate_fn(batch):
    # создание тензоров
    inputs = [torch.tensor(item['input_ids']) for item in batch]
    labels = [torch.tensor(item['labels']) for item in batch]
    # длины входных последовательностей
    lengths = torch.tensor([len(x) for x in inputs])

    # выравнивание паддингами
    inputs = pad_sequence(inputs, batch_first=True, padding_value=PAD_TOKEN_ID)
    labels = pad_sequence(labels, batch_first=True, padding_value=PAD_TOKEN_ID)

    # перенос на GPU (при наличии)
    return  {
        'input_ids': inputs.to(get_device()),
        'labels': labels.to(get_device()),
        'lengths': lengths.cpu() # это должно быть на CPU
    }

In [ ]:
# создаем обучающий и валидационный даталоадеры
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# инициализация модели
model = SequencePredictionLSTM(
    vocab_size=len(tokenizer.get_vocab()),
    embedding_dim=LSTM_HIDDEN_STATE,
    hidden_size=LSTM_HIDDEN_STATE,
    padding_idx=PAD_TOKEN_ID
)
model.to(get_device()) # перенос на GPU (при наличии)

In [ ]:
# обучение
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN_ID)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau( # для уменьшения шага если метрика не будет улучшаться
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)
# старт обучения (с выводом LOSS и ROUGE на каждой эпохе)
train(
    model=model,
    n_epochs=EPOCHS,
    tokenizer=tokenizer,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    max_length=max_length_with_eos,
    pad_token_id=PAD_TOKEN_ID
)

In [ ]:
# сохранение весов обученной модели
save_model_dict(model)

In [ ]:
# генерация примеров текстов
lstm_text_examples: pd.DataFrame = lstm_generate_text_examples(
    model, tokenizer, val_dataloader, max_length, PAD_TOKEN_ID
)

In [ ]:
# просмотр в случайном режиме
lstm_text_examples.sample(7)

## Предобученная модель из библиотеки transformers

In [ ]:
# чистка текстов для трансформера
val_texts = val_texts['text'].apply(clean_string_transformer)

# удаление коротких текстов (< 3 слов)
val_texts = drop_short_texts(val_texts, min_length)

# удаление длинных текстов (> 26 слов)
val_texts = drop_long_texts(val_texts, max_length)

In [ ]:
# проверка длин
val_texts.str.split().apply(len).describe()

In [ ]:
# разбивка на input (3/4), reference (1/4)
val_texts = val_texts.apply(split_text)

val_data = {
    'input_ids': [row[0] for row in val_texts],
    'references': [row[1] for row in val_texts]
}

In [ ]:
# так как у используемой модели нет pad_token, но есть eos_token (50256)
# padding будет заполняться токеном eos
tokenizer.pad_token = tokenizer.eos_token

# для корректного автопродолжения токены eos должны быть слева
tokenizer.padding_side = "left"

In [ ]:
# обработка батча (токенизация и перенос на GPU)
def collate_fn(batch):
    inputs = [item["input_ids"] for item in batch]
    references = [item["references"] for item in batch]
    tokenized = tokenizer(
        inputs,
        padding=True,
        return_tensors='pt',
    )

    return {
        'input_ids': tokenized['input_ids'].to(get_device()),
        'attention_mask': tokenized['attention_mask'].to(get_device()),
        'references': references
    }

In [ ]:
# для валидации создаем датасет ("коробочный" из библиотеки datasets) и даталоадер
val_dataset = Dataset.from_dict(val_data)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# инициализация класса модели
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# сообщаем модели, что eos токен это padding
model.config.pad_token_id = tokenizer.eos_token_id
# перенос на GPU
model.to(get_device())

In [ ]:
# оценка по rouge_1
rouge_1 = evaluate_rouge_transformer(model, tokenizer, val_dataloader, max_length)
print('ROUGE 1: ', rouge_1)

In [ ]:
# примеры сгенерированных текстов
transformer_text_examples: pd.DataFrame = transformer_generate_text_examples(
    model, tokenizer, val_dataloader, max_length
)

In [ ]:
# выводим 7 случайных
transformer_text_examples.sample(7)